# 010 Time Travel

这是 LangGraph 学习线的第十份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langgraph/use-time-travel

学习目标：

1. 理解 time travel 依赖 checkpoint history
2. 学会用 `get_state_history(...)` 找到历史 checkpoint
3. 学会用 `invoke(None, checkpoint.config)` replay 后续节点
4. 学会用 `update_state(...)` 从旧 checkpoint fork 新分支
5. 理解 replay 会重新触发 LLM、API、interrupt 等后续节点副作用
6. 对比 time travel 和本仓库 Harness ledger / approval resume

## 1. Time Travel 解决什么问题

Time travel 不是让你读取历史日志这么简单。

它有两个核心能力：

| 能力 | 含义 |
| --- | --- |
| Replay | 从历史 checkpoint 重新执行后续节点 |
| Fork | 从历史 checkpoint 改一部分 state，然后走出新分支 |

重要边界：

```text
checkpoint 之前的节点不会重新执行。
checkpoint 之后的节点会重新执行。
```

所以如果后续节点包含 LLM、API、数据库写入、interrupt，它们都会再次触发。

In [ ]:
import importlib.metadata

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from typing_extensions import TypedDict

print("langgraph", importlib.metadata.version("langgraph"))

## 2. 定义一个两节点 graph

这个 graph 很简单：

```text
START -> generate_topic -> write_joke -> END
```

我们用 `node_runs` 记录节点实际执行次数。

这样可以看清 replay 和 fork 到底重跑了哪些节点。

In [ ]:
class TravelState(TypedDict):
    topic: str
    joke: str


node_runs = {"generate_topic": 0, "write_joke": 0}


def generate_topic(state: TravelState) -> dict:
    node_runs["generate_topic"] += 1
    print("generate_topic run:", node_runs["generate_topic"])
    return {"topic": "socks in the dryer"}


def write_joke(state: TravelState) -> dict:
    node_runs["write_joke"] += 1
    print("write_joke run:", node_runs["write_joke"])
    return {
        "joke": (
            "joke about " + state["topic"] + " | run " + str(node_runs["write_joke"])
        )
    }


travel_builder = StateGraph(TravelState)
travel_builder.add_node("generate_topic", generate_topic)
travel_builder.add_node("write_joke", write_joke)

travel_builder.add_edge(START, "generate_topic")
travel_builder.add_edge("generate_topic", "write_joke")
travel_builder.add_edge("write_joke", END)

travel_graph = travel_builder.compile(checkpointer=InMemorySaver())
travel_graph

## 3. 第一次执行并生成 checkpoint history

Time travel 必须依赖 checkpointer。

没有 checkpoint history，就没有地方可以回到过去。

In [ ]:
travel_config = {"configurable": {"thread_id": "time-travel-demo"}}

first_result = travel_graph.invoke(
    {"topic": "", "joke": ""},
    travel_config,
)

print("first result:", first_result)
print("node runs:", node_runs)

## 4. 查看 checkpoint history

`get_state_history(config)` 返回的是 state snapshot 列表。

注意：history 是倒序的，也就是最新 checkpoint 在前。

In [ ]:
history = list(travel_graph.get_state_history(travel_config))

for snapshot in history:
    print(
        "step=", snapshot.metadata.get("step"),
        "next=", snapshot.next,
        "values=", snapshot.values,
        "checkpoint_id=", snapshot.config["configurable"].get("checkpoint_id"),
    )

## 5. Replay：从 write_joke 前面重新执行

我们找这个 checkpoint：

```text
next == ("write_joke",)
```

这表示：

```text
generate_topic 已经完成，下一步准备执行 write_joke。
```

从这里 replay 时，`generate_topic` 不会重跑，`write_joke` 会重跑。

In [ ]:
before_joke = next(snapshot for snapshot in history if snapshot.next == ("write_joke",))

replay_result = travel_graph.invoke(None, before_joke.config)

print("replay result:", replay_result)
print("node runs after replay:", node_runs)

## 6. Replay final checkpoint 是 no-op

如果从最终 checkpoint replay：

```text
next == ()
```

说明没有后续节点可执行，所以 replay 基本等于读取最终结果。

In [ ]:
latest_history = list(travel_graph.get_state_history(travel_config))
final_checkpoint = next(snapshot for snapshot in latest_history if snapshot.next == ())

no_op_result = travel_graph.invoke(None, final_checkpoint.config)

print("no-op replay result:", no_op_result)
print("node runs after no-op replay:", node_runs)

## 7. Fork：从旧 checkpoint 修改 state 走新分支

Fork 不会回滚旧历史。

它会基于某个旧 checkpoint 创建一个新的 checkpoint 分支。

下面从 `before_joke` 这个 checkpoint 改 `topic`，再继续执行 `write_joke`。

In [ ]:
fork_config = travel_graph.update_state(
    before_joke.config,
    values={"topic": "chickens"},
    as_node="generate_topic",
)

fork_snapshot = travel_graph.get_state(fork_config)
print("fork snapshot values:", fork_snapshot.values)
print("fork snapshot next:", fork_snapshot.next)

fork_result = travel_graph.invoke(None, fork_config)
print("fork result:", fork_result)
print("node runs after fork:", node_runs)

## 8. Fork 之后历史不会消失

Fork 是分支，不是 rollback。

原来的 socks 分支和新的 chickens 分支都会留在 checkpoint history 里。

In [ ]:
branch_history = list(travel_graph.get_state_history(travel_config))

for snapshot in branch_history[:8]:
    print(
        "step=", snapshot.metadata.get("step"),
        "next=", snapshot.next,
        "topic=", snapshot.values.get("topic"),
        "joke=", snapshot.values.get("joke"),
    )

## 9. Time travel 遇到 interrupt

如果 replay 或 fork 的后续节点里有 `interrupt()`，它会再次暂停。

也就是说：

```text
time travel 不会复用旧的人类回答。
后续 interrupt 会重新触发，等待新的 Command(resume=...)。
```

In [ ]:
class AskState(TypedDict):
    name: str
    greeting: str


def ask_name(state: AskState) -> dict:
    answer = interrupt("请输入姓名")
    return {"name": answer}


def greet(state: AskState) -> dict:
    return {"greeting": "你好，" + state["name"]}


ask_graph = (
    StateGraph(AskState)
    .add_node("ask_name", ask_name)
    .add_node("greet", greet)
    .add_edge(START, "ask_name")
    .add_edge("ask_name", "greet")
    .add_edge("greet", END)
    .compile(checkpointer=InMemorySaver())
)

ask_config = {"configurable": {"thread_id": "time-travel-interrupt-demo"}}

ask_first = ask_graph.invoke({"name": "", "greeting": ""}, ask_config)
print("first interrupt:", ask_first["__interrupt__"][0].value)

ask_final = ask_graph.invoke(Command(resume="Alice"), ask_config)
print("final after Alice:", ask_final)

In [ ]:
ask_history = list(ask_graph.get_state_history(ask_config))
before_ask = [snapshot for snapshot in ask_history if snapshot.next == ("ask_name",)][-1]

ask_replay = ask_graph.invoke(None, before_ask.config)
print("replay interrupt:", ask_replay["__interrupt__"][0].value)

ask_bob = ask_graph.invoke(Command(resume="Bob"), ask_config)
print("final after Bob branch:", ask_bob)

## 10. 和 Harness 的关系

| LangGraph Time Travel | Harness 风格智能体 |
| --- | --- |
| checkpoint history | ledger / run state history |
| replay from checkpoint | 从某个 run state 继续执行 |
| fork from checkpoint | 基于旧观察修改计划后走新分支 |
| interrupt re-trigger | approval 重新发起等待 |

关键差异：

```text
LangGraph 把 checkpoint / replay / fork 做成标准 runtime 能力。
Harness 如果要做到同等能力，需要自己设计 run state 持久化、ledger、恢复点和分支规则。
```

## 11. 本讲练习

请判断下面场景应该用 replay 还是 fork：

1. 模型生成结果不稳定，想从调用模型前重新跑一次。
2. 发现中间 state 的 `topic` 写错了，想改成另一个值再继续。
3. 某个工具短暂失败，想从工具节点前重试。
4. 想保留原始执行结果，同时探索另一个 human answer。

参考答案：

1. Replay
2. Fork
3. Replay
4. Fork

## 12. 本讲小结

这一讲的核心：

```text
Time travel = checkpoint history + replay + fork。
```

你现在应该能看懂：

- `get_state_history(config)`
- `checkpoint.config`
- `invoke(None, checkpoint.config)`
- `update_state(checkpoint.config, values=...)`
- `as_node` 为什么会影响后续从哪里继续
- replay 和 fork 的副作用风险
- time travel 遇到 interrupt 为什么会重新暂停

下一讲可以继续学习 Memory。